# 02 — Data cleaning

Per `PROJECT_PLAN.md` Section 3 this notebook stands in for four originals:
`data-preprocessing-flats`, the (never-uploaded) houses equivalent,
`merge-flats-and-house`, and `data-preprocessing-level-2`.

**What actually exists in `src/` today is the first stage only:**
`clean_flats()` and `clean_houses()` in `src/preprocessing/cleaning.py`. They
turn the raw scrapes into the shared `*_cleaned.csv` schema. This notebook runs
them and checks them against the committed interim files.

**Not yet ported — a real gap:** the *merge* step
(`flats_cleaned.csv` + `house_cleaned.csv` → `gurgaon_properties.csv`, 3 961 rows)
and the *level-2* step (`gurgaon_properties.csv` → `gurgaon_properties_cleaned_v1.csv`,
3 803 rows, 20 → 17 cols). Neither is in `src/`, so the numbered pipeline
currently has two untraced steps between here and `06_feature_engineering`
(whose input is `cleaned_v1.csv`). This is the same "every file should have one
traceable notebook" issue flagged for the `_v2` step in Section 1.

This is a **demonstration** of already-tested code — no new logic.

In [1]:
import sys, logging
from pathlib import Path

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

# clean_flats / clean_houses log every row they drop - surface it.
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

from src.preprocessing.cleaning import clean_flats, clean_houses

RAW = REPO_ROOT / "data" / "raw"
INTERIM = REPO_ROOT / "data" / "interim"


def compare(name, got, expected):
    # row-count + per-column exact-match check against a committed CSV
    got = got.reset_index(drop=True)
    print(f"{name}: got {got.shape}, expected {expected.shape}, "
          f"row counts {'match' if len(got) == len(expected) else 'DIFFER'}")
    assert list(got.columns) == list(expected.columns), "column mismatch"
    n = min(len(got), len(expected))
    deviations = {}
    total = match = 0
    for col in expected.columns:
        a, b = got[col].iloc[:n], expected[col].iloc[:n]
        if b.dtype.kind in "fi":
            m = np.isclose(pd.to_numeric(a, errors="coerce").astype(float),
                           b.astype(float), equal_nan=True)
        else:
            m = a.astype(str) == b.astype(str)
        total += len(m); match += int(m.sum())
        if not m.all():
            deviations[col] = int((~m).sum())
    print(f"  cell match: {match}/{total} = {100 * match / total:.3f}%")
    if deviations:
        print(f"  columns that differ: {deviations}")
    return deviations

## Flats — `clean_flats(raw_flats)`

`data/raw/flats.csv` → the cleaned-flats schema. What it does:

- drop `link`, `property_id`
- rename `area` (a `₹.../sq.ft.` string) → `price_per_sqft`; parse it to a number
- strip the `4.2 ★` rating suffix off `society`, lower-case it
- drop `price == "Price on Request"` rows, then parse `price` to crore (`Lac` → ÷100)
- drop rows with no `bedRoom`; take the leading integer of `bedRoom` / `bathroom` / `balcony` (`"No"` → `0`)
- `additionalRoom` → lower-case, null → `"not available"`
- `floorNum` via `_parse_floor_num` (`"Ground"` → 0, `"Basement"` → **-1**, `"4 of 12"` → 4)
- `facing` null → `"NA"`
- derive `area` = `round(price × 1e7 / price_per_sqft)`; tag `property_type = "flat"`

In [2]:
raw_flats = pd.read_csv(RAW / "flats.csv")
flats = clean_flats(raw_flats).reset_index(drop=True)
print("raw     :", raw_flats.shape)
print("cleaned :", flats.shape)

clean_flats: dropped 11 'Price on Request' rows


clean_flats: dropped 9 rows with no bedRoom


raw     : (3017, 20)
cleaned : (2997, 20)


In [3]:
flats.head()

,property_name,property_type,society,price,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating
0,2 BHK Flat in Krishna Colony,flat,maa bhagwati residency,0.45,5000.0,900.0,Carpet area: 900 (83.61 sq.m.),2,2,1,not available,"Krishna Colony, Gurgaon, Haryana",4,West,1 to 5 Year Old,"['Chintapurni Mandir', 'State bank ATM', 'Pear...",So with lift.Maa bhagwati residency is one of ...,"['3 Fan', '4 Light', '1 Wardrobe', 'No AC', 'N...","['Feng Shui / Vaastu Compliant', 'Security / F...","['Environment4 out of 5', 'Safety4 out of 5', ..."
1,2 BHK Flat in Ashok Vihar,flat,apna enclave,0.50,7692.0,650.0,Carpet area: 650 (60.39 sq.m.),2,2,1,not available,"46b, Ashok Vihar, Gurgaon, Haryana",1,West,10+ Year Old,"['Chintapurni Mandir', 'Sheetla Mata Mandir', ...","Property situated on main road, railway statio...","['3 Wardrobe', '4 Fan', '1 Exhaust Fan', '1 Ge...","['Security / Fire Alarm', 'Maintenance Staff',...","['Environment4 out of 5', 'Safety4 out of 5', ..."
2,2 BHK Flat in Sohna,flat,tulsiani easy in homes,0.40,6722.0,595.0,Carpet area: 595 (55.28 sq.m.),2,2,3,not available,"Sohna, Gurgaon, Haryana",12,NA,0 to 1 Year Old,"['Huda City Metro', 'Golf Course extn road', '...","This property is 15 km away from badshapur, gu...",NaN,"['Power Back-up', 'Feng Shui / Vaastu Complian...","['Environment4 out of 5', 'Safety4 out of 5', ..."
3,2 BHK Flat in Sector 61 Gurgaon,flat,smart world orchard,1.47,12250.0,1200.0,Carpet area: 1200 (111.48 sq.m.),2,2,2,study room,"Sector 61 Gurgaon, Gurgaon, Haryana",2,NA,Dec 2023,"['Sector 55-56 Metro station', 'Bestech Centra...",Near to metro station of sector 56 and opposit...,NaN,"['Security / Fire Alarm', 'Private Garden / Te...",NaN
4,2 BHK Flat in Sector 92 Gurgaon,flat,parkwood westend,0.70,5204.0,1345.0,Super Built up area 1345(124.95 sq.m.),2,2,3,study room,"Sector 92 Gurgaon, Gurgaon, Haryana",5,NA,Under Construction,"['Yadav Clinic', 'Bangali Clinic', 'Dr. J. S. ...",We are the proud owners of this 2 bhk alongwit...,[],NaN,"['Environment5 out of 5', 'Safety3 out of 5', ..."


In [4]:
flats_expected = pd.read_csv(INTERIM / "flats_cleaned.csv")
flats_dev = compare("clean_flats vs flats_cleaned.csv", flats, flats_expected)

clean_flats vs flats_cleaned.csv: got (2997, 20), expected (2997, 20), row counts match
  cell match: 59063/59940 = 98.537%
  columns that differ: {'floorNum': 3, 'facing': 874}


In [5]:
# What the two differing columns actually are:
fe = flats_expected
mism_facing = flats["facing"].astype(str) != fe["facing"].astype(str)
print("facing  :", int(mism_facing.sum()), "rows -",
      "got", flats.loc[mism_facing, "facing"].unique().tolist(),
      "vs expected", fe.loc[mism_facing, "facing"].unique().tolist())

mism_floor = ~np.isclose(pd.to_numeric(flats["floorNum"], errors="coerce").astype(float),
                         fe["floorNum"].astype(float), equal_nan=True)
print("floorNum:", int(mism_floor.sum()), "rows -",
      "got", flats.loc[mism_floor, "floorNum"].tolist(),
      "vs expected", fe.loc[mism_floor, "floorNum"].tolist())

facing  : 874 rows - got ['NA'] vs expected [nan]
floorNum: 3 rows - got ['-1', '-1', '-1'] vs expected [1.0, 1.0, 1.0]


### The two flats deviations — both deliberate

`clean_flats` matches `flats_cleaned.csv` on **row count and every cell** except:

1. **`facing` — 874 rows.** The module fills missing `facing` with the string
   `"NA"`; the committed file left them as real `NaN`. An explicit sentinel vs a
   null — cosmetic, chosen for clarity.
2. **`floorNum` — 3 rows.** These are basement listings. The original
   notebook's floor parser extracted digits *after* replacing text, stripping
   the sign so `"-1"` became `1` — a basement indistinguishable from a 1st-floor
   flat. `_parse_floor_num` keeps the sign (`r"(-?\d+)"`), so these three come
   out as **-1**. This is the fix, not a regression — see the `cleaning.py`
   docstring.

## Houses — `clean_houses(raw_houses)`

**Reconstructed**, not ported: the houses-cleaning notebook was never provided.
`clean_houses` mirrors `clean_flats` against the houses schema:

- `rate` plays the role flats' relabeled `area` does (both `₹.../sq.ft.` strings) → `price_per_sqft`
- `noOfFloor` → `floorNum`
- **`society` null → `"independent"`** *before* any string work — a house with no
  society is a normal standalone house, not an error
- everything else (price / bedroom / balcony / floor / facing / derived `area`) as in `clean_flats`
- also a `drop_duplicates()` on the raw rows first; tag `property_type = "house"`

In [6]:
raw_houses = pd.read_csv(RAW / "houses.csv")
houses = clean_houses(raw_houses).reset_index(drop=True)
print("raw     :", raw_houses.shape)
print("cleaned :", houses.shape)

clean_houses: dropped 21 fully-duplicated raw rows


clean_houses: dropped 10 'Price on Request' rows


clean_houses: dropped 49 rows with no bedRoom


raw     : (1044, 21)
cleaned : (964, 20)


In [7]:
houses.head()

,property_name,property_type,society,price,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating
0,5 Bedroom House for sale in Sector 70A Gurgaon,house,bptp visionnaire,5.25,20115.0,2610.0,Plot area 290(242.48 sq.m.),5,4,3+,servant room,"29b, Sector 70A Gurgaon, Gurgaon, Haryana",3,North-East,0 to 1 Year Old,"['Good Earth City Center 2', 'Kunskapsskolan I...",Do you wish to buy an independent house in sec...,"['1 Wardrobe', '1 Fan', '1 Exhaust Fan', '1 Ge...","['Feng Shui / Vaastu Compliant', 'Private Gard...","['Environment5 out of 5', 'Lifestyle4 out of 5..."
1,5 Bedroom House for sale in Sector 21A Faridabad,house,independent,5.70,105751.0,539.0,Plot area 539(50.07 sq.m.),5,4,2,"store room,pooja room,servant room","Sector 21A Faridabad, Gurgaon, Haryana",2,NA,5 to 10 Year Old,NaN,"Hi, we have an independent house/villa availab...","['1 Water Purifier', '5 Fan', '1 Exhaust Fan',...","['Private Garden / Terrace', 'Park', 'Visitor ...",NaN
2,10 Bedroom House for sale in Sushant Lok Phase 1,house,independent,2.10,38251.0,549.0,Plot area 61(51 sq.m.),10,10,3+,servant room,"Sushant Lok Phase 1, Gurgaon, Haryana",5,West,0 to 1 Year Old,"['Sector 42-43 metro station', 'Huda city cent...","Monthly rental income is rs1,40,000/- Best opt...","['10 Bed', '3 Fan', '10 Geyser', '2 Light', 'N...","['Maintenance Staff', 'Water Storage', 'Visito...","['Environment5 out of 5', 'Lifestyle5 out of 5..."
3,21 Bedroom House for sale in Sector 54 Gurgaon,house,independent,5.00,43066.0,1161.0,Plot area 129(107.86 sq.m.),21,21,3+,servant room,"Sector 54 Gurgaon, Gurgaon, Haryana",5,North,1 to 5 Year Old,"['Sector 53-54 metro station', 'Sector 54 chow...","129 sq yd plot size. 5 floors built up , fully...","['1 Water Purifier', '21 Fan', '1 Fridge', '1 ...","['Feng Shui / Vaastu Compliant', 'Private Gard...","['Environment4 out of 5', 'Lifestyle5 out of 5..."
4,12 Bedroom House for sale in Sushant Lok Phase 1,house,independent,3.00,53763.0,558.0,Plot area 62(51.84 sq.m.),12,12,3+,others,"1228, Sushant Lok Phase 1, Gurgaon, Haryana",5,West,Within 6 months,"['Sector 42-43 metro station', 'Huda city cent...",Best for investment purpose. Fully furnished b...,"['1 Water Purifier', '1 Fridge', '12 Fan', '1 ...","['Maintenance Staff', 'Water Storage', 'Visito...","['Environment5 out of 5', 'Lifestyle5 out of 5..."


In [8]:
houses_expected = pd.read_csv(INTERIM / "house_cleaned.csv")
houses_dev = compare("clean_houses vs house_cleaned.csv", houses, houses_expected)

clean_houses vs house_cleaned.csv: got (964, 20), expected (964, 20), row counts match
  cell match: 18973/19280 = 98.408%
  columns that differ: {'society': 4, 'facing': 303}


In [9]:
he = houses_expected
mism_soc = houses["society"].astype(str) != he["society"].astype(str)
pd.DataFrame({
    "clean_houses (fixed)": houses.loc[mism_soc, "society"].values,
    "house_cleaned.csv (buggy)": he.loc[mism_soc, "society"].values,
})

,clean_houses (fixed),house_cleaned.csv (buggy)
0,surendra homes dayanand colony,surendra homes dayaindependentd colony
1,"anand garden, sector-105","aindependentd garden, sector-105"
2,anant raj estates,aindependentt raj estates
3,bestech park view ananda,bestech park view aindependentda


### The houses deviations

- **`facing` — 303 rows.** Same `"NA"`-vs-`NaN` sentinel choice as flats.
- **`society` — 4 rows.** The committed `house_cleaned.csv` was built by casting
  `society` to string and substring-replacing `"nan"` → `"independent"`, which
  also mangled real names containing the letters *n-a-n*:
  `dayanand` → `dayaindependentd`, `anand` → `aindependentd`,
  `anant` → `aindependentt`, `ananda` → `aindependentda`. `clean_houses` fills
  the actual null values first, so these four keep their real names. Also a
  documented fix.

`clean_houses` is otherwise row-for-row identical to the committed file, but it
has **no source notebook to have been verified against** — treat it as
"matches the one output we have" rather than "known correct".

## What this notebook does *not* cover

The cleaned-flats and cleaned-houses frames above still have to be **merged**
and put through a **level-2** pass before `06_feature_engineering` can consume
them:

| step | in → out | shape | in `src/`? |
|---|---|---|---|
| merge | `flats_cleaned` + `house_cleaned` → `gurgaon_properties.csv` | → (3961, 20) | **no** |
| level-2 | `gurgaon_properties.csv` → `gurgaon_properties_cleaned_v1.csv` | (3961, 20) → (3803, 17) | **no** |

Both `merge-flats-and-house.ipynb` and `data-preprocessing-level-2.ipynb` exist
under `notebooks_original/` but have not been ported to `src/`. Until they are,
the numbered pipeline can't be run end-to-end from raw — it picks up again at
`cleaned_v1.csv`, which `06_feature_engineering` takes as its input.